In [4]:
# Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,Corner Kicks,Free Kicks,Offsides
# Away Team,1,19,7,2,0,10,17,3
# Home Team,2,13,18,2,0,7,7,3

# Key to results data:
# FTHG and HG = Full Time Home Team Goals
# FTAG and AG = Full-Time Away Team Goals
# FTR and Res = Full-Time Result (H=Home Win, D=Draw, A=Away Win)

# Match Statistics (where available)
# HS = Home Team Shots
# AS = Away Team Shots
# HC = Home Team Corners
# AC = Away Team Corners
# HF = Home Team Fouls Committed
# AF = Away Team Fouls Committed
# HO = Home Team Offsides
# AO = Away Team Offsides
# HY = Home Team Yellow Cards
# AY = Away Team Yellow Cards
# HR = Home Team Red Cards
# AR = Away Team Red Cards

import pandas as pd
import os

# --- Configuration ---
DATA_DIR = './LiveSum_++/english-premier-league/' 
print(f"Using data directory: {DATA_DIR}")
# Define the stats you are looking for in a match
# AWAY TEAM STATS:
# Away Team,1,19,7,2,0,10,17,3  (Goals, Shots, Fouls, Yellow Cards, Red Cards, Corner Kicks, Free Kicks, Offsides)
# HOME TEAM STATS:
# Home Team,2,13,18,2,0,7,7,3   (Goals, Shots, Fouls, Yellow Cards, Red Cards, Corner Kicks, Free Kicks, Offsides)
TARGET_STATS = {
    # Away Team Conditions
    'FTAG': {'operator': '==', 'value': 1},     # Full Time Away Goals == 1
    'AS': {'operator': '==', 'value': 19},      # Away Shots == 19
    'AF': {'operator': '==', 'value': 7},       # Away Fouls == 7
    'AY': {'operator': '==', 'value': 2},       # Away Yellow Cards == 2
    'AR': {'operator': '==', 'value': 0},       # Away Red Cards == 0
    'AC': {'operator': '==', 'value': 10},      # Away Corner Kicks == 10
    # Note: 'Away Free Kicks' and 'Away Offsides' are typically not available as direct columns.
    # If they are in your files, use their exact column names here.

    # Home Team Conditions
    'FTHG': {'operator': '==', 'value': 2},     # Full Time Home Goals == 2
    'HS': {'operator': '==', 'value': 13},      # Home Shots == 13
    'HF': {'operator': '==', 'value': 18},      # Home Fouls == 18
    'HY': {'operator': '==', 'value': 2},       # Home Yellow Cards == 2
    'HR': {'operator': '==', 'value': 0},       # Home Red Cards == 0
    'HC': {'operator': '==', 'value': 7},       # Home Corner Kicks == 7
    # Note: 'Home Free Kicks' and 'Home Offsides' are typically not available as direct columns.
    # If they are in your files, use their exact column names here.
}

# List of CSV files to process
CSV_FILES = [f'{year}-{str(int(year) + 1)[-2:]}.csv' 
             for year in range(2013, 2022)] # Generates '2013-14.csv', '2014-15.csv', etc. up to '2021-22.csv'
print(f"CSV files to process: {CSV_FILES}")
# --- The rest of the code (load_and_combine_data and find_matches_by_stats functions) remains the same ---

def load_and_combine_data(data_directory, csv_files):
    all_data = []
    for filename in csv_files:
        filepath = os.path.join(data_directory, filename)
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath, encoding='latin1') 
                df['Season'] = filename.split('.')[0] 
                all_data.append(df)
                print(f"Successfully loaded: {filename}")
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        else:
            print(f"File not found: {filename}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data loaded. Please check your DATA_DIR and CSV_FILES list.")
        return pd.DataFrame()

def find_matches_by_stats(dataframe, target_stats):
    if dataframe.empty:
        print("Input DataFrame is empty. Cannot filter.")
        return pd.DataFrame()

    condition = pd.Series(True, index=dataframe.index)

    for col, criteria in target_stats.items():
        if col in dataframe.columns:
            operator = criteria['operator']
            value = criteria['value']
            
            if operator == '>':
                condition &= (dataframe[col] > value)
            elif operator == '<':
                condition &= (dataframe[col] < value)
            elif operator == '==':
                condition &= (dataframe[col] == value)
            elif operator == '>=':
                condition &= (dataframe[col] >= value)
            elif operator == '<=':
                condition &= (dataframe[col] <= value)
            elif operator == '!=':
                condition &= (dataframe[col] != value)
            else:
                print(f"Unsupported operator '{operator}' for column '{col}'. Skipping this condition.")
        else:
            print(f"Column '{col}' not found in DataFrame. Skipping this condition.")
            
    return dataframe[condition]

# --- Execute the functions ---

combined_football_data = load_and_combine_data(DATA_DIR, CSV_FILES)

if not combined_football_data.empty:
    print(f"\nTotal rows loaded: {len(combined_football_data)}")
    print("First 5 rows of combined data:")
    display(combined_football_data.head()) 

    matching_matches = find_matches_by_stats(combined_football_data, TARGET_STATS)

    print(f"\nFound {len(matching_matches)} match(es) matching the specified stats:")
    display(matching_matches) 
else:
    print("No data to process. Please check the data loading step.")

Using data directory: ./LiveSum_++/english-premier-league/
CSV files to process: ['2013-14.csv', '2014-15.csv', '2015-16.csv', '2016-17.csv', '2017-18.csv', '2018-19.csv', '2019-20.csv', '2020-21.csv', '2021-22.csv']
Successfully loaded: 2013-14.csv
Successfully loaded: 2014-15.csv
Successfully loaded: 2015-16.csv
Successfully loaded: 2016-17.csv
Successfully loaded: 2017-18.csv
Successfully loaded: 2018-19.csv
Successfully loaded: 2019-20.csv
Successfully loaded: 2020-21.csv
Successfully loaded: 2021-22.csv

Total rows loaded: 3080
First 5 rows of combined data:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
0,E0,17/08/13,Arsenal,Aston Villa,1,3,A,1,1,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,17/08/13,Liverpool,Stoke,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,17/08/13,Norwich,Everton,2,2,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,17/08/13,Sunderland,Fulham,0,1,A,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,17/08/13,Swansea,Man United,1,4,A,0,2,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Found 1 match(es) matching the specified stats:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
235,E0,01/02/14,Stoke,Man United,2,1,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
